# Case Study 2: RAG Pipeline — Technical Documentation Q&A

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TechyNilesh/DeepTextSearch/blob/main/notebooks/02_case_study_rag_pipeline.ipynb)

**Goal**: Build a complete Retrieval Augmented Generation pipeline for answering questions from technical documentation.

**Features covered**: Full RAG pipeline (retrieve → rerank → generate), multiple embedding models, tiny vs large rerankers, FAISS index types (flat/hnsw), `RerankRequest`, `TextSearchTool` for agents, model presets listing

In [1]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
!pip install -q git+https://github.com/TechyNilesh/DeepTextSearch.git

## Step 1: Build a Knowledge Base

Simulating technical documentation chunks about vector databases and search.

In [3]:
knowledge_base = [
    # FAISS
    "FAISS (Facebook AI Similarity Search) is a library for efficient similarity search and clustering of dense vectors. It supports billion-scale datasets and GPU acceleration.",
    "FAISS provides multiple index types: IndexFlatIP for exact search, IndexIVFFlat for approximate search with inverted file indexes, and IndexHNSWFlat for graph-based approximate search.",
    "FAISS IndexHNSW builds a hierarchical navigable small world graph. It offers the best query-time performance but uses more memory and has slower index build time.",
    # Embedding Models
    "BGE-M3 is a multilingual embedding model from BAAI that supports over 100 languages. It produces 1024-dimensional dense vectors and also supports sparse embeddings for hybrid search.",
    "Sentence transformers convert text into fixed-size dense vector embeddings that capture semantic meaning. They use a bi-encoder architecture for fast inference.",
    "Cross-encoder models jointly encode query-passage pairs and produce a relevance score. They are more accurate than bi-encoders but much slower since they cannot pre-compute embeddings.",
    # Search Methods
    "BM25 (Best Matching 25) is a probabilistic retrieval function that scores documents based on term frequency, inverse document frequency, and document length normalization.",
    "Hybrid search combines dense vector retrieval with sparse keyword matching (BM25) using fusion techniques. Reciprocal Rank Fusion (RRF) is the most common fusion method.",
    "Reciprocal Rank Fusion (RRF) combines ranked lists from multiple retrieval systems by computing 1/(k+rank) for each document, where k is a constant (typically 60).",
    # Vector Databases
    "ChromaDB is an open-source vector database optimized for AI applications. It stores embeddings with metadata and supports persistent local storage.",
    "Qdrant is a vector similarity search engine written in Rust. It supports advanced metadata filtering, hybrid search, and can run as a local service or cloud deployment.",
    "PostgreSQL with the pgvector extension enables vector similarity search directly within a relational database, allowing you to combine vector search with SQL queries.",
    # RAG
    "Retrieval Augmented Generation (RAG) is a technique that enhances LLM responses by retrieving relevant context from a knowledge base before generating an answer.",
    "A typical RAG pipeline has three stages: retrieval (finding relevant documents), reranking (refining results with a cross-encoder), and generation (feeding context to an LLM).",
    "Chunking strategies for RAG include fixed-size chunks, sentence-based splitting, paragraph-based splitting, and recursive character splitting. Chunk size affects retrieval quality.",
    # Reranking
    "The ms-marco-MiniLM-L-6-v2 cross-encoder is a popular reranking model. It has ~90 MB size and provides the best speed/quality trade-off for English text.",
    "TinyBERT-L-2-v2 is an ultra-small cross-encoder reranker at only ~17 MB. It is ideal for CPU-only deployments, serverless functions, and edge devices.",
    "BGE-Reranker-v2-m3 is a multilingual cross-encoder that supports 100+ languages with up to 8192 token context length for long document reranking.",
]

print(f"Knowledge base: {len(knowledge_base)} chunks")

Knowledge base: 18 chunks


## Step 2: Explore Available Models

In [4]:
from DeepTextSearch import EMBEDDING_PRESETS, RERANKER_PRESETS

print("=== Embedding Models ===")
for name, info in EMBEDDING_PRESETS.items():
    print(f"  {name}")
    print(f"    {info['dimensions']}d | ~{info['size_mb']} MB | {info['languages']} | {info['description']}")

print("\n=== Reranker Models ===")
for name, info in RERANKER_PRESETS.items():
    print(f"  {name}")
    print(f"    ~{info['size_mb']} MB | {info['languages']} | {info['description']}")

=== Embedding Models ===
  BAAI/bge-m3
    1024d | ~2200 MB | 100+ | Best multilingual. Highest quality, dense+sparse support.
  sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
    384d | ~470 MB | 50+ | Fast multilingual. Good quality/speed trade-off.
  intfloat/multilingual-e5-large
    1024d | ~2240 MB | 100+ | Strong multilingual from Microsoft. High quality.
  BAAI/bge-small-en-v1.5
    384d | ~130 MB | English | Tiny & fast. Great for prototyping.
  BAAI/bge-base-en-v1.5
    768d | ~440 MB | English | Balanced quality and speed.
  BAAI/bge-large-en-v1.5
    1024d | ~1340 MB | English | Highest quality English model.
  sentence-transformers/all-MiniLM-L6-v2
    384d | ~90 MB | English | Ultra lightweight. Fastest inference.
  thenlper/gte-small
    384d | ~67 MB | English | Alibaba GTE. Tiny but powerful.
  nomic-ai/nomic-embed-text-v1.5
    768d | ~548 MB | English | Long context (8K tokens). Open source.

=== Reranker Models ===
  cross-encoder/ms-marco-TinyBERT-L-2-

## Step 3: Index with Different Configurations

In [5]:
from DeepTextSearch import TextEmbedder, TextSearch, Reranker
import time

# Small & fast model with flat index
print("=== Indexing with bge-small-en (flat) ===")
start = time.time()
embedder = TextEmbedder(
    model_name="BAAI/bge-small-en-v1.5",
    vector_store="faiss",
    index_type="flat",
)
embedder.index(knowledge_base)
print(f"  Time: {time.time()-start:.1f}s | {embedder.corpus_size} docs | {embedder.dimension}d | Device: {embedder.device}")

# Same model with HNSW index (for large-scale use)
print("\n=== Indexing with bge-small-en (hnsw) ===")
start = time.time()
embedder_hnsw = TextEmbedder(
    model_name="BAAI/bge-small-en-v1.5",
    vector_store="faiss",
    index_type="hnsw",
)
embedder_hnsw.index(knowledge_base)
print(f"  Time: {time.time()-start:.1f}s | {embedder_hnsw.corpus_size} docs | {embedder_hnsw.dimension}d")

=== Indexing with bge-small-en (flat) ===


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  Time: 3.9s | 18 docs | 384d | Device: cuda

=== Indexing with bge-small-en (hnsw) ===


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  Time: 3.5s | 18 docs | 384d


## Step 4: RAG Pipeline — Retrieve → Rerank → Generate

In [6]:
search = TextSearch(embedder, mode="hybrid")

def rag_search(query, top_n_retrieve=10, top_n_final=3):
    """Complete RAG retrieval pipeline."""
    # Stage 1: Hybrid search (broad retrieval)
    candidates = search.search(query, top_n=top_n_retrieve)
    
    # Stage 2: Rerank (precision refinement)
    reranker = Reranker(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")
    reranked = reranker.rerank_search_results(query, candidates, top_n=top_n_final)
    
    return reranked

# Test with multiple queries
queries = [
    "What is the difference between FAISS flat and HNSW index?",
    "How does hybrid search work?",
    "What is the smallest reranker model available?",
    "How to do vector search in PostgreSQL?",
]

for query in queries:
    print(f"\nQ: {query}")
    results = rag_search(query)
    for i, r in enumerate(results, 1):
        print(f"  {i}. [{r['score']:.4f}] {r['text'][:100]}...")


Q: What is the difference between FAISS flat and HNSW index?


  1. [4.0088] FAISS IndexHNSW builds a hierarchical navigable small world graph. It offers the best query-time per...
  2. [0.2977] FAISS provides multiple index types: IndexFlatIP for exact search, IndexIVFFlat for approximate sear...
  3. [-0.3091] FAISS (Facebook AI Similarity Search) is a library for efficient similarity search and clustering of...

Q: How does hybrid search work?


  1. [7.6053] Hybrid search combines dense vector retrieval with sparse keyword matching (BM25) using fusion techn...
  2. [1.0787] Qdrant is a vector similarity search engine written in Rust. It supports advanced metadata filtering...
  3. [-0.9071] BGE-M3 is a multilingual embedding model from BAAI that supports over 100 languages. It produces 102...

Q: What is the smallest reranker model available?


  1. [4.3225] TinyBERT-L-2-v2 is an ultra-small cross-encoder reranker at only ~17 MB. It is ideal for CPU-only de...
  2. [3.6513] The ms-marco-MiniLM-L-6-v2 cross-encoder is a popular reranking model. It has ~90 MB size and provid...
  3. [0.2347] BGE-Reranker-v2-m3 is a multilingual cross-encoder that supports 100+ languages with up to 8192 toke...

Q: How to do vector search in PostgreSQL?


  1. [6.3013] PostgreSQL with the pgvector extension enables vector similarity search directly within a relational...
  2. [-3.8455] Hybrid search combines dense vector retrieval with sparse keyword matching (BM25) using fusion techn...
  3. [-3.8647] Qdrant is a vector similarity search engine written in Rust. It supports advanced metadata filtering...


## Step 5: Build LLM Prompt from Retrieved Context

In [7]:
query = "What is the recommended approach for building a RAG pipeline?"
context_docs = rag_search(query, top_n_retrieve=15, top_n_final=5)

# Build prompt
context = "\n".join([f"- {doc['text']}" for doc in context_docs])

prompt = f"""You are a helpful technical assistant. Answer the question based on the provided context.

Context:
{context}

Question: {query}

Answer:"""

print(prompt)
print("\n" + "="*60)
print("Pass this prompt to any LLM (OpenAI, Anthropic, local model, etc.)")

You are a helpful technical assistant. Answer the question based on the provided context.

Context:
- A typical RAG pipeline has three stages: retrieval (finding relevant documents), reranking (refining results with a cross-encoder), and generation (feeding context to an LLM).
- Retrieval Augmented Generation (RAG) is a technique that enhances LLM responses by retrieving relevant context from a knowledge base before generating an answer.
- Chunking strategies for RAG include fixed-size chunks, sentence-based splitting, paragraph-based splitting, and recursive character splitting. Chunk size affects retrieval quality.
- BGE-M3 is a multilingual embedding model from BAAI that supports over 100 languages. It produces 1024-dimensional dense vectors and also supports sparse embeddings for hybrid search.
- BM25 (Best Matching 25) is a probabilistic retrieval function that scores documents based on term frequency, inverse document frequency, and document length normalization.

Question: What 

## Step 6: Compare Tiny vs Standard Reranker

In [8]:
from DeepTextSearch import RerankRequest

query = "best vector database for production"
candidates = search.search(query, top_n=10)
passages = [r.to_dict() for r in candidates]

# Tiny reranker (~17 MB)
print("=== Tiny Reranker (TinyBERT, ~17 MB) ===")
reranker_tiny = Reranker("cross-encoder/ms-marco-TinyBERT-L-2-v2")
start = time.time()
request = RerankRequest(query=query, passages=passages.copy())
ranked_tiny = reranker_tiny.rerank(request, top_n=3)
tiny_time = time.time() - start
for r in ranked_tiny:
    print(f"  [{r['score']:.4f}] {r['text'][:80]}...")
print(f"  Time: {tiny_time*1000:.0f}ms")

# Standard reranker (~90 MB)
print("\n=== Standard Reranker (MiniLM-L-6, ~90 MB) ===")
reranker_std = Reranker("cross-encoder/ms-marco-MiniLM-L-6-v2")
start = time.time()
request = RerankRequest(query=query, passages=passages.copy())
ranked_std = reranker_std.rerank(request, top_n=3)
std_time = time.time() - start
for r in ranked_std:
    print(f"  [{r['score']:.4f}] {r['text'][:80]}...")
print(f"  Time: {std_time*1000:.0f}ms")

=== Tiny Reranker (TinyBERT, ~17 MB) ===


  [-2.2738] ChromaDB is an open-source vector database optimized for AI applications. It sto...
  [-5.2984] PostgreSQL with the pgvector extension enables vector similarity search directly...
  [-8.3858] Qdrant is a vector similarity search engine written in Rust. It supports advance...
  Time: 7ms

=== Standard Reranker (MiniLM-L-6, ~90 MB) ===


  [1.0980] ChromaDB is an open-source vector database optimized for AI applications. It sto...
  [-6.8661] PostgreSQL with the pgvector extension enables vector similarity search directly...
  [-6.9466] BGE-M3 is a multilingual embedding model from BAAI that supports over 100 langua...
  Time: 8ms


## Step 7: Agent Tool Interface

Use DeepTextSearch as a callable tool for AI agents.

In [9]:
import json
from DeepTextSearch.agents import TextSearchTool

# Create agent tool
tool = TextSearchTool(
    embedder,
    reranker=Reranker("cross-encoder/ms-marco-TinyBERT-L-2-v2"),
)

# Call it like a function
result_json = tool("how does BM25 scoring work?", k=3)
results = json.loads(result_json)

print("=== Agent Tool Results ===")
for r in results:
    print(f"  [{r['score']:.4f}] {r['text'][:80]}...")

# Show the function-calling schema
print("\n=== Tool Definition (OpenAI/Claude compatible) ===")
print(json.dumps(tool.tool_definition, indent=2))

=== Agent Tool Results ===
  [7.4648] BM25 (Best Matching 25) is a probabilistic retrieval function that scores docume...
  [0.6831] Hybrid search combines dense vector retrieval with sparse keyword matching (BM25...
  [-9.9476] BGE-M3 is a multilingual embedding model from BAAI that supports over 100 langua...

=== Tool Definition (OpenAI/Claude compatible) ===
{
  "type": "function",
  "function": {
    "name": "search_texts",
    "description": "Search a text corpus using semantic and keyword hybrid search. Returns the most relevant text passages for a given query.",
    "parameters": {
      "type": "object",
      "properties": {
        "query": {
          "type": "string",
          "description": "The search query text."
        },
        "k": {
          "type": "integer",
          "description": "Number of results to return.",
          "default": 5
        },
        "mode": {
          "type": "string",
          "enum": [
            "auto",
            "hybrid",
      

## Step 8: Encode Without Indexing

Get raw embeddings for custom downstream tasks.

In [10]:
import numpy as np

vectors = embedder.encode([
    "vector database comparison",
    "how to build a search engine",
    "machine learning tutorial",
])

print(f"Shape: {vectors.shape}")
print(f"Dtype: {vectors.dtype}")

# Compute cosine similarity manually
sim = np.dot(vectors[0], vectors[1])
print(f"\nSimilarity('vector database' vs 'search engine'): {sim:.4f}")
sim2 = np.dot(vectors[0], vectors[2])
print(f"Similarity('vector database' vs 'ML tutorial'): {sim2:.4f}")

Shape: (3, 384)
Dtype: float32

Similarity('vector database' vs 'search engine'): 0.5736
Similarity('vector database' vs 'ML tutorial'): 0.6818


---

**Summary**: This case study demonstrated a complete RAG pipeline with model comparison, reranker benchmarking, agent tool integration, and raw embedding access.